# Gradient Matching Poison Attack — Pipeline Explorer

Walk through the complete poisoning pipeline step by step.  
Modify the **Experiment Switcher** (Section 8) to swap recipes, losses, and
hyperparameters, then re-run from Section 3 onward.

**Pipeline:**  
1. Config & Setup  
2. Data (Kettle)  
3. Victim Model  
4. Clean Surrogate Training  
5. Poison Crafting (Witch)  
6. Visualize Poisons  
7. Validation  
8. Experiment Switcher  
9. Privacy Extension (Laplace / Gaussian gradient noise)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import io
import re
import time
from contextlib import redirect_stdout

%matplotlib inline

import forest
from forest import utils
from forest.victims.laplace_mechanism import add_laplace_noise, log_gradient_stats

print('PyTorch version:', torch.__version__)
print('CUDA available: ', torch.cuda.is_available())

## Section 1: Configuration

`forest.options().parse_args([])` loads all CLI defaults.  
Override specific flags inline — no need to touch the command line.

In [ ]:
args = forest.options().parse_args([])

# ─── MAIN KNOBS ──────────────────────────────────────────────────────────
args.net         = ['ResNet18']       # 'ResNet18', 'MobileNetV2', 'VGG11'
args.dataset     = 'CIFAR10'          # 'CIFAR10', 'CIFAR100', 'MNIST'
args.recipe      = 'gradient-matching'  # 'gradient-matching', 'poison-frogs', 'bullseye'
args.eps         = 16                 # L-inf bound in pixel space / 255
args.budget      = 0.01               # fraction of training data to poison
args.targets     = 1                  # number of target images
args.restarts    = 2                  # attack restarts (8 = paper results)
args.attackiter  = 50                 # gradient-matching steps per restart
args.attackoptim = 'signAdam'         # 'signAdam', 'Adam', 'PGD', 'GD', 'momPGD'
args.loss        = 'similarity'       # 'similarity', 'cosine1', 'SE', 'MSE'
args.poisonkey   = '2000000000'       # reproducibility seed (None = random)
args.vruns       = 1                  # validation re-runs
args.dryrun      = False              # True = 1-batch smoke test
args.optimization = 'conservative'    # 'conservative', 'private', 'basic'
args.data_path   = '~/data'
# ─────────────────────────────────────────────────────────────────────────

torch.backends.cudnn.benchmark = forest.consts.BENCHMARK
setup = utils.system_startup(args)
print()
print('Device:', setup['device'])

## Section 2: Data — Kettle

`Kettle` manages all dataset logic:
- `trainset / validset` — full clean data
- `poisonset` — the subset that will be perturbed
- `targetset` — the image(s) the attack tries to misclassify
- `poison_lookup` — maps `image_id → slice_index` in `poison_delta`
- `poison_delta` — **only the perturbations** (added on-the-fly during training)

In [ ]:
# Temporary victim just to extract batch_size and augmentation config
_tmp = forest.Victim(args, setup=setup)

data = forest.Kettle(args, _tmp.defs.batch_size, _tmp.defs.augmentations, setup=setup)
del _tmp

print(f'Train set:   {len(data.trainset):,} images')
print(f'Valid set:   {len(data.validset):,} images')
print(f'Poison set:  {len(data.poisonset):,} images  ({args.budget * 100:.1f}% of train)')
print(f'Target set:  {len(data.targetset)} image(s)')
print()
print('Poison setup:')
for k, v in data.poison_setup.items():
    print(f'  {k}: {v}')
print()
_d0 = data.initialize_poison()
print(f'poison_delta shape: {_d0.shape}  [N_poison x C x H x W]')
print(f'poison_lookup: {len(data.poison_lookup)} entries')
print(f'  first 3: {dict(list(data.poison_lookup.items())[:3])}')

In [ ]:
class_names = data.trainset.classes

dm = data.dm.cpu().view(-1)   # channel means (C,)
ds = data.ds.cpu().view(-1)   # channel stds  (C,)

def denorm(t):
    return torch.clamp(t.cpu() * ds[:, None, None] + dm[:, None, None], 0, 1)

# ── Target image(s) ──────────────────────────────────────────────────────
n_t = len(data.targetset)
fig, axes = plt.subplots(1, n_t, figsize=(4 * n_t, 4))
if n_t == 1:
    axes = [axes]
for i, (img, lbl, _) in enumerate(data.targetset):
    intended_list = data.poison_setup['intended_class']
    intended_i = intended_list[i] if isinstance(intended_list, list) else intended_list
    axes[i].imshow(denorm(img).permute(1, 2, 0).numpy())
    axes[i].set_title(
        f'TARGET\nTrue: {class_names[lbl]}\nIntended: {class_names[intended_i]}',
        fontsize=10)
    axes[i].axis('off')
plt.suptitle('Target image — should be misclassified after poisoning', fontsize=11)
plt.tight_layout()
plt.show()

# ── Sample poison candidates (clean) ─────────────────────────────────────
n_show = min(10, len(data.poisonset))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    if i < n_show:
        img, lbl, _ = data.poisonset[i]
        ax.imshow(denorm(img).permute(1, 2, 0).numpy())
        ax.set_title(class_names[lbl], fontsize=9)
    ax.axis('off')
plt.suptitle('Poison Candidates — clean images before the attack', fontsize=12)
plt.tight_layout()
plt.show()

## Section 3: Victim Model

Initializes the surrogate model, optimizer, and scheduler.  
Re-run this cell for a fresh random seed.

In [ ]:
model = forest.Victim(args, setup=setup)
defs  = model.defs

total_p     = sum(p.numel() for p in model.model.parameters())
trainable_p = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f'Architecture:     {args.net[0]}')
print(f'Total params:     {total_p:,}')
print(f'Trainable params: {trainable_p:,}')
print()
print('Training strategy (defs):')
print(f'  epochs:        {defs.epochs}')
print(f'  batch_size:    {defs.batch_size}')
print(f'  lr:            {defs.lr}')
print(f'  optimizer:     {defs.optimizer}')
print(f'  scheduler:     {defs.scheduler}')
print(f'  augmentations: {defs.augmentations}')
print(f'  privacy:       {defs.privacy}')
print(f'  validate every {defs.validate} epoch(s)')

## Section 4: Clean Surrogate Training

Train on **clean** data (no poisoning).  
The gradient of `CE(model(target), intended_class)` on this trained model  
is the **target gradient** the attack will try to replicate via `poison_delta`.

In [ ]:
print(f'Training {args.net[0]} on clean data ({defs.epochs} epochs)...')
print()

t0 = time.time()
stats_clean = model.train(data, max_epoch=args.max_epoch)
print(f'\nDone in {time.time() - t0:.1f}s')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ep = list(range(len(stats_clean['train_losses'])))

ax = axes[0]
ax.plot(ep, stats_clean['train_losses'], label='Train')
if stats_clean.get('valid_losses'):
    ax.plot(ep, stats_clean['valid_losses'], label='Valid', linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Loss')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(ep, [a * 100 for a in stats_clean['train_accs']], label='Train')
if stats_clean.get('valid_accs'):
    ax.plot(ep, [a * 100 for a in stats_clean['valid_accs']], label='Valid', linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)'); ax.set_title('Accuracy')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
if stats_clean.get('target_accs'):
    ax.plot(ep, [a * 100 for a in stats_clean['target_accs']],
            color='red', label='Fool acc (target->intended)')
    ax.plot(ep, [a * 100 for a in stats_clean['target_accs_clean']],
            color='green', linestyle='--', label='Clean acc (target->true)')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'No target log', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)'); ax.set_title('Target on Clean Model')
ax.grid(alpha=0.3)

plt.suptitle('Surrogate Model Training (clean data)', fontsize=13)
plt.tight_layout()
plt.show()

_tacc = stats_clean['train_accs'][-1] * 100
print(f'Final train acc: {_tacc:.1f}%')
if stats_clean.get('valid_accs'):
    _vacc = stats_clean['valid_accs'][-1] * 100
    print(f'Final valid acc: {_vacc:.1f}%')

## Section 5: Poison Crafting — Witch (Gradient Matching)

`_brew()` outer loop:
1. Precompute **target gradient** — `∇_θ CE(model(target), intended_class)`
2. For each restart: initialize `poison_delta`, run `attackiter` steps:
   - Compute `poison_grad = ∇_θ CE(model(poison+δ), poison_labels)`
   - Minimize **passenger loss** `= −cosine(poison_grad, target_grad)`
   - Project `δ` back into `[−ε/255, +ε/255]` L∞ ball per channel
3. Select restart with lowest passenger loss

**Lower passenger loss = better gradient alignment = stronger attack.**

In [ ]:
witch = forest.Witch(args, setup=setup)

print(f'Recipe:       {args.recipe}')
print(f'Loss:         {args.loss}')
print(f'Optimizer:    {args.attackoptim}')
print(f'Iterations:   {args.attackiter} per restart')
print(f'Restarts:     {args.restarts}')
print(f'Eps:          {args.eps} / 255  (L-inf per channel)')
print(f'Budget:       {args.budget * 100:.1f}%  =  {len(data.poisonset)} poison images')

In [ ]:
_brew_buf = io.StringIO()

print('Brewing (output captured for later parsing)...\n')
t0 = time.time()
with redirect_stdout(_brew_buf):
    poison_delta = witch.brew(model, data)
brew_elapsed = time.time() - t0

raw_brew_output = _brew_buf.getvalue()
print(raw_brew_output)

print(f'Brewing time: {brew_elapsed:.1f}s')
print(f'Optimal passenger loss: {witch.stat_optimal_loss:.4e}')
print(f'poison_delta shape: {poison_delta.shape}')

_ds_mean = data.ds.cpu().mean().item()
_max_norm = poison_delta.abs().max().item()
print(f'Max |delta| (normalized): {_max_norm:.4f}')
print(f'Max |delta| (pixels):     {_max_norm * _ds_mean * 255:.2f}  vs eps={args.eps}')

In [ ]:
# Parse per-iteration passenger losses from the captured brew output
# Expected print format: 'Iteration X: Target loss is Y.YYYY, ...'
_iter_re = re.compile(r'Iteration\s+(\d+):\s+Target loss is\s+([0-9.e+-]+)')

restart_curves = []
_current  = []
_prev_step = -1

for line in raw_brew_output.split('\n'):
    m = _iter_re.search(line)
    if m:
        step = int(m.group(1))
        loss = float(m.group(2))
        if step <= _prev_step:   # step reset = new restart
            restart_curves.append(_current)
            _current = []
        _current.append((step, loss))
        _prev_step = step
if _current:
    restart_curves.append(_current)

if restart_curves:
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, curve in enumerate(restart_curves):
        steps, losses = zip(*curve)
        ax.plot(steps, losses, marker='o', markersize=3,
                label=f'Restart {i + 1}', alpha=0.85)
    _opt = witch.stat_optimal_loss
    ax.set_xlabel('Attack Iteration')
    ax.set_ylabel('Passenger Loss  (lower = better alignment)')
    ax.set_title(f'Gradient Matching Loss per Restart  [optimal: {_opt:.4e}]')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Could not parse iteration losses.')
    print('Run with dryrun=False and attackiter > 4.')

## Section 6: Visualize Poisons

- **Row 1:** Clean image  
- **Row 2:** Perturbation δ normalized to [0, 1] for visibility (actual magnitude is imperceptible)  
- **Row 3:** Poisoned image = clean + δ

In [ ]:
n_show = min(5, len(data.poisonset))
lookup_items = list(data.poison_lookup.items())[:n_show]

fig, axes = plt.subplots(3, n_show, figsize=(3 * n_show, 9))
if n_show == 1:
    axes = axes.reshape(3, 1)

for col, (img_id, slice_idx) in enumerate(lookup_items):
    clean_img, lbl, _ = data.poisonset[col]
    delta = poison_delta[slice_idx].cpu()
    poisoned_img = clean_img + delta

    axes[0, col].imshow(denorm(clean_img).permute(1, 2, 0).numpy())
    axes[0, col].set_title(class_names[lbl], fontsize=9)
    axes[0, col].axis('off')

    d_vis = (delta - delta.min()) / (delta.max() - delta.min() + 1e-8)
    axes[1, col].imshow(d_vis.permute(1, 2, 0).numpy())
    axes[1, col].set_title(f'max={delta.abs().max().item():.3f}', fontsize=9)
    axes[1, col].axis('off')

    axes[2, col].imshow(denorm(poisoned_img).permute(1, 2, 0).numpy())
    axes[2, col].set_title('poisoned', fontsize=9)
    axes[2, col].axis('off')

for r, rlbl in enumerate(['Clean', 'Perturbation\n(normalized)', 'Poisoned']):
    axes[r, 0].set_ylabel(rlbl, fontsize=10, rotation=0, labelpad=65, va='center')

plt.suptitle('Clean  |  Perturbation (normalized)  |  Poisoned', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
_delta_cpu = poison_delta.detach().cpu()
_n = len(data.poisonset)

flat_px = _delta_cpu.numpy().flatten() * 255
linf    = _delta_cpu.view(_n, -1).abs().max(dim=1).values.numpy() * 255
l2      = _delta_cpu.view(_n, -1).norm(dim=1).numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.hist(flat_px, bins=80, alpha=0.75, color='steelblue', edgecolor='none')
ax.axvline( args.eps, color='red', linestyle='--', label=f'+eps={args.eps}')
ax.axvline(-args.eps, color='red', linestyle='--', label=f'-eps={args.eps}')
ax.set_xlabel('Perturbation (pixel scale)'); ax.set_ylabel('Count')
ax.set_title('Value Distribution'); ax.legend()

ax = axes[1]
ax.hist(linf, bins=30, alpha=0.75, color='darkorange', edgecolor='none')
ax.axvline(args.eps, color='red', linestyle='--', label=f'eps={args.eps}')
ax.set_xlabel('L-inf norm (pixels)'); ax.set_title('Per-Image L-inf Norm'); ax.legend()

ax = axes[2]
ax.hist(l2, bins=30, alpha=0.75, color='seagreen', edgecolor='none')
ax.set_xlabel('L2 norm (normalized space)'); ax.set_title('Per-Image L2 Norm')

plt.suptitle('Poison Perturbation Statistics', fontsize=13)
plt.tight_layout()
plt.show()

status = 'OK' if linf.max() <= args.eps + 0.5 else 'EXCEEDED'
print(f'Constraint: max L-inf = {linf.max():.2f} px  vs  eps={args.eps}  [{status}]')
print(f'Mean L2 per image: {l2.mean():.4f}')

## Section 7: Validation

Fresh model init → train on poisoned data → measure attack success.

- **Fool acc** (red) — target classified as intended (wrong) class → attack success rate  
- **Valid acc** (blue) — overall accuracy → collateral damage check (should stay high)

In [ ]:
print(f'Validating: {args.vruns} fresh init(s) trained on poisoned data...')
print()

t0 = time.time()
stats_val = model.validate(data, poison_delta)
print(f'\nDone in {time.time() - t0:.1f}s')

print('\n=== Attack Results ===')
for key in ['train_accs', 'valid_accs', 'target_accs', 'target_accs_clean',
            'train_losses', 'valid_losses', 'target_losses']:
    if stats_val.get(key):
        print(f'  {key:30s}: {stats_val[key][-1]:.4f}')

if stats_val.get('target_accs'):
    _fool = stats_val['target_accs'][-1] * 100
    _vacc = stats_val.get('valid_accs', [0])[-1] * 100
    print(f'\n  Attack SUCCESS (fool acc):   {_fool:.1f}%')
    print(f'  Model intact  (valid acc):   {_vacc:.1f}%')

In [ ]:
if stats_val.get('target_accs'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ep_v = list(range(len(stats_val['train_losses'])))

    ax = axes[0]
    ax.plot(ep_v, [a * 100 for a in stats_val['valid_accs']],
            label='Valid acc', color='steelblue')
    ax.plot(ep_v, [a * 100 for a in stats_val['train_accs']],
            label='Train acc', color='steelblue', linestyle='--', alpha=0.6)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.set_title('Overall Accuracy (poisoned model)'); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(ep_v, [a * 100 for a in stats_val['target_accs']],
            color='red', linewidth=2, label='Fool acc  (target -> intended)')
    ax.plot(ep_v, [a * 100 for a in stats_val['target_accs_clean']],
            color='green', linestyle='--', label='Clean acc (target -> true)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.set_title('Target Classification (poisoned model)'); ax.legend(); ax.grid(alpha=0.3)
    ax.set_ylim([-5, 105])

    _fool = stats_val['target_accs'][-1] * 100
    plt.suptitle(
        f'Validation  recipe={args.recipe}  eps={args.eps}  budget={args.budget}'  
        f'  fool={_fool:.1f}%',
        fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('No target_accs in stats_val. Check targetset is non-empty.')

## Section 8: Experiment Switcher

Change any value, then re-run **from Section 3 (Victim) downward**.

| Parameter | Options |
|---|---|
| `recipe` | `gradient-matching`, `poison-frogs`, `bullseye`, `watermarking` |
| `loss` | `similarity`, `cosine1`, `SE`, `MSE`, `scalar_product` |
| `attackoptim` | `signAdam`, `Adam`, `PGD`, `GD`, `momPGD` |
| `eps` | 8 (weak) / 16 (default) / 32 (strong) |
| `budget` | 0.005 / 0.01 (default) / 0.05 |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  EXPERIMENT SWITCHER — edit, then re-run from Section 3 downward
# ═══════════════════════════════════════════════════════════════════
EXP = {
    'net':         ['ResNet18'],
    'recipe':      'gradient-matching',
    'dataset':     'CIFAR10',
    'eps':         16,
    'budget':      0.01,
    'loss':        'similarity',
    'attackoptim': 'signAdam',
    'attackiter':  50,
    'restarts':    2,
    'poisonkey':   '2000000000',
    'vruns':       1,
}

for k, v in EXP.items():
    setattr(args, k, v)

print('Args updated:')
for k, v in EXP.items():
    print(f'  args.{k:15s} = {v}')
print()
print('Re-run from SECTION 3 (Victim) downward.')

## Section 9: Privacy Extension — Laplace / Gaussian Gradient Noise

`PrivacyStrategy` applies per-batch:
1. **Gradient clipping** — clips each parameter's gradient to L2 norm ≤ `clip`
2. **Noise injection** — adds Laplace or Gaussian noise scaled by `clip × noise`

This makes it harder for the attacker's gradient alignment to survive into the model weights.  
Compare **passenger loss** with vs. without DP — higher under privacy = weaker attack.  

Source: [forest/victims/laplace_mechanism.py](forest/victims/laplace_mechanism.py)

In [ ]:
args_priv = forest.options().parse_args([])

# Copy base settings
for attr in ['net', 'dataset', 'recipe', 'eps', 'budget', 'targets',
             'restarts', 'attackiter', 'attackoptim', 'loss',
             'poisonkey', 'vruns', 'dryrun', 'data_path']:
    setattr(args_priv, attr, getattr(args, attr))

# Privacy overrides
args_priv.optimization   = 'private'
args_priv.gradient_noise = 0.01   # noise scale σ
args_priv.gradient_clip  = 1.0   # L2 clip bound C
args_priv.noise_type     = 'laplace'  # 'gaussian' or 'laplace'
args_priv.log_gradients  = True

print('Privacy config:')
print(f'  optimization:   {args_priv.optimization}')
print(f'  noise_type:     {args_priv.noise_type}')
print(f'  gradient_clip:  {args_priv.gradient_clip}  (L2 clip bound C)')
print(f'  gradient_noise: {args_priv.gradient_noise}  (noise scale sigma)')
print(f'  log_gradients:  {args_priv.log_gradients}')
print()
_eff_noise = args_priv.gradient_clip * args_priv.gradient_noise
print(f'Effective noise std = C x sigma = {_eff_noise}')

In [ ]:
print('Training PRIVATE surrogate (with gradient noise)...\n')

model_priv = forest.Victim(args_priv, setup=setup)
stats_priv = model_priv.train(data, max_epoch=args_priv.max_epoch)

witch_priv = forest.Witch(args_priv, setup=setup)
_buf_priv  = io.StringIO()
with redirect_stdout(_buf_priv):
    poison_delta_priv = witch_priv.brew(model_priv, data)
print(_buf_priv.getvalue())

print('-' * 50)
_std_loss  = witch.stat_optimal_loss
_priv_loss = witch_priv.stat_optimal_loss
print(f'Passenger loss — standard:   {_std_loss:.4e}')
print(f'Passenger loss — private:    {_priv_loss:.4e}')
_ratio = _priv_loss / (_std_loss + 1e-12)
print(f'Ratio private / standard:    {_ratio:.2f}x')
print()
if _ratio > 1.0:
    print('DP is working: gradient alignment is harder under privacy noise.')
else:
    print('Ratio <= 1.0: try increasing gradient_noise or gradient_clip.')

In [ ]:
if stats_priv and stats_priv.get('gradient_log'):
    grad_log = stats_priv['gradient_log']
    param_names = [k for k in grad_log[0] if k not in ('step', 'epoch')]

    # log_gradient_stats is called twice per batch: before and after noise
    before_log = grad_log[0::2]
    after_log  = grad_log[1::2]
    show_params = param_names[:5]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    for pname in show_params:
        norms = [e.get(pname, 0) for e in before_log]
        axes[0].plot(norms, label=pname, alpha=0.8)
    axes[0].set_title('Gradient L2 Norms BEFORE Laplace Noise')
    axes[0].set_ylabel('L2 Norm'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

    for pname in show_params:
        norms = [e.get(pname, 0) for e in after_log]
        axes[1].plot(norms, label=pname, alpha=0.8)
    axes[1].set_title('Gradient L2 Norms AFTER Laplace Noise')
    axes[1].set_xlabel('Batch Step'); axes[1].set_ylabel('L2 Norm')
    axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

    plt.suptitle('Per-Parameter Gradient Norms — Private Training', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No gradient_log found in stats_priv.')
    print('Ensure args_priv.log_gradients = True and args_priv.optimization = "private".')